# Task 2 | Data quality and cleanup

Run from the top to refresh the six CSVs in `data/cleaned/` from the original Excel workbook. Use a Python kernel with pandas and openpyxl. If openpyxl is missing, run `%pip install openpyxl` in that kernel.

The exports keep the full extract. Reporting windows, metric filters, and channel mapping belong in the analysis notebooks. The final table records the data issues and their effects.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "raw" / "Data & Insights - Analyst Case.xlsx").is_file()
)
WORKBOOK = ROOT / "data" / "raw" / "Data & Insights - Analyst Case.xlsx"
CLEANED = ROOT / "data" / "cleaned"
Q3_START = "2026-07-01"
Q3_END = "2026-10-01"
QA_NAME_PATTERNS = {
    "qa lucky", "test user", "test qa", "test test",
    "تجربه تطبيق", "تجربة",
}
CLEANED.mkdir(exist_ok=True)

In [2]:
# Read the six data sheets from the original workbook.
sheets = pd.read_excel(
    WORKBOOK,
    sheet_name=["users", "transactions", "loans", "plus_subscriptions",
                "cvm_campaigns", "marketing_spend"],
    engine="openpyxl",
)
users_raw = sheets["users"]
transactions_raw = sheets["transactions"]
loans_raw = sheets["loans"]
plus_subscriptions_raw = sheets["plus_subscriptions"]
cvm_campaigns_raw = sheets["cvm_campaigns"]
marketing_spend_raw = sheets["marketing_spend"]

In [ ]:
# Users: find exact copies and derive QA IDs from the reviewed name patterns.
duplicate_users = users_raw.duplicated()
unique_users = users_raw.loc[~duplicate_users]
normalized_names = unique_users["full_name"].str.strip().str.casefold()
qa_mask = normalized_names.isin(QA_NAME_PATTERNS)
qa_users = unique_users.loc[qa_mask, ["user_id", "full_name"]]
qa_user_ids = qa_users["user_id"].unique()
users = unique_users.loc[~qa_mask].copy()

# Standardize UTM strings while keeping the original values for traceability.
users["utm_source_normalized"] = users["utm_source"].str.strip().str.lower().replace("", pd.NA)
users["utm_campaign_normalized"] = users["utm_campaign"].str.strip().str.lower().replace("", pd.NA)

print(f"Exact duplicate users removed: {duplicate_users.sum():,} / {len(users_raw):,}")
print(f"QA users removed: {len(qa_user_ids):,}")
display(qa_users.sort_values("user_id"))
print(f"Clean users: {len(users):,}")
users.to_csv(CLEANED / "users.csv", index=False)


# Review input for a semantic pass over every user.
transaction_counts = transactions_raw.groupby("user_id")["txn_id"].nunique()
qa_review = unique_users[["user_id", "full_name"]].copy()
qa_review["distinct_transaction_count"] = qa_review["user_id"].map(transaction_counts).fillna(0).astype(int)
qa_review.to_csv(CLEANED / "qa_user_review.csv", index=False)

Exact duplicate users removed: 0 / 6,230
QA users removed: 30


,user_id,full_name
6082,106201,QA Lucky
3796,106202,test test
4714,106203,QA Lucky
4121,106204,Test User
1681,106205,تجربه تطبيق
3076,106206,QA Lucky
796,106207,test qa
1726,106208,Test User
3399,106209,QA Lucky
4249,106210,Test User


Clean users: 6,200


In [ ]:
# Transactions: remove exact copies first so QA rows are counted once.
duplicate_transactions = transactions_raw.duplicated()
unique_transactions = transactions_raw.loc[~duplicate_transactions]
qa_transaction_mask = unique_transactions["user_id"].isin(qa_user_ids)
transactions = unique_transactions.loc[~qa_transaction_mask].copy()

print(f"Exact copies removed: {duplicate_transactions.sum():,} / {len(transactions_raw):,}")
print(f"QA rows removed: {qa_transaction_mask.sum():,} / {len(unique_transactions):,} deduplicated rows")
print(f"Clean transactions: {len(transactions):,}")
transactions.to_csv(CLEANED / "transactions.csv", index=False)


# Save the first four steps of the Task 3.1 deposit bridge.
q3_deposits = transactions_raw.loc[
    transactions_raw["txn_type"].eq("deposit")
    & transactions_raw["txn_ts"].ge(Q3_START)
    & transactions_raw["txn_ts"].lt(Q3_END)
]
successful_deposits = q3_deposits.loc[q3_deposits["status"].eq("success")]
unique_deposits = successful_deposits.drop_duplicates()
valid_deposits = unique_deposits.loc[unique_deposits["user_id"].isin(users["user_id"])]
audit_steps = [
    ("Raw UTC, all statuses", q3_deposits),
    ("Raw UTC, successful", successful_deposits),
    ("Remove exact copies", unique_deposits),
    ("Keep valid non-QA users", valid_deposits),
]
deposit_audit = pd.DataFrame(
    [(step, label, len(rows), rows["amount_egp"].sum())
     for step, (label, rows) in enumerate(audit_steps, start=1)],
    columns=["step", "scope", "deposits", "volume_egp"],
)
deposit_audit.to_csv(CLEANED / "q3_deposit_audit.csv", index=False)

Exact copies removed: 577 / 42,378
QA rows removed: 682 / 41,801 deduplicated rows
Clean transactions: 41,119


In [5]:
# Loans: profiling found no rows needing removal.
loans = loans_raw.copy()
print("Blank last repayment dates are preserved.")
print(f"Clean loans: {len(loans):,} / {len(loans_raw):,} rows")
loans.to_csv(CLEANED / "loans.csv", index=False)

Blank last repayment dates are preserved.
Clean loans: 1,310 / 1,310 rows


In [6]:
# Plus subscriptions: profiling found no rows needing removal.
plus_subscriptions = plus_subscriptions_raw.copy()
print("Blank end dates indicate ongoing subscriptions.")
print(f"Clean plus subscriptions: {len(plus_subscriptions):,} / {len(plus_subscriptions_raw):,} rows")
plus_subscriptions.to_csv(CLEANED / "plus_subscriptions.csv", index=False)

Blank end dates indicate ongoing subscriptions.
Clean plus subscriptions: 111 / 111 rows


In [7]:
# Cvm campaigns: profiling found no rows needing removal.
cvm_campaigns = cvm_campaigns_raw.copy()
print("Blank send timestamps are expected for holdout users.")
print(f"Clean cvm campaigns: {len(cvm_campaigns):,} / {len(cvm_campaigns_raw):,} rows")
cvm_campaigns.to_csv(CLEANED / "cvm_campaigns.csv", index=False)

Blank send timestamps are expected for holdout users.
Clean cvm campaigns: 697 / 697 rows


In [8]:
# Marketing spend: profiling found no rows needing removal.
marketing_spend = marketing_spend_raw.copy()
print("Platform-reported installs are preserved as supplied.")
print(f"Clean marketing spend: {len(marketing_spend):,} / {len(marketing_spend_raw):,} rows")
marketing_spend.to_csv(CLEANED / "marketing_spend.csv", index=False)

Platform-reported installs are preserved as supplied.
Clean marketing spend: 276 / 276 rows


## Task 2 summary

| Issue | Detection | Affected | Handling and impact |
|---|---|---|---|
| Apparent QA accounts | Review names and transaction counts per user, then match six reviewed name patterns | 30 users and 682 transactions after deduplication | Exclude by user ID. Removes EGP 38,590,000 from successful UTC Q3 deposits after deduplication. |
| Exact transaction copies | Compare every field | 577 of 42,378 raw rows | Keep one copy. Removes EGP 991,800 from successful UTC Q3 deposits. |
| UTM string variations | Compare before and after trimming and lowercasing | 1,274 clean users | Preserve original values and add normalized columns for attribution. |
| Missing UTM source | Count blank or null sources | 794 raw users | Keep missing attribution. |

The local QA review CSV includes every user ID and name, with distinct transactions counted by user ID. Users with no transactions have a count of zero. Use it for an exhaustive semantic or AI-assisted review of possible test names, then validate candidates before adding to the reviewed patterns. The notebook applies the reviewed patterns deterministically.

The users and transactions exports have 6,200 and 41,119 rows. Loans, Plus, CVM and marketing retain all 1,310, 111, 697 and 276 rows respectively. Preserve expected blanks such as KYC dates, open Plus end dates and holdout send times. These figures describe the supplied extract.

### Acquisition channel mapping used in analysis

Match the normalized source to these aliases. Retain Organic, Referral, and Direct as separate labels when useful. The marketing comparison can group those three under Other channels. Blank sources remain missing attribution. Any other unrecognized value can be reviewed separately.

| Channel | Normalized source aliases |
|---|---|
| Meta | meta, facebook ads, facebook, fb, instagram, ig |
| Google Ads | google, google ads, adwords, gads |
| TikTok | tiktok, tt |
| Referral | referral, ref, invite |
| Organic | organic |
| Direct | (direct) |

The detailed profiling found zero missing keys, conflicting rows sharing a key after exact deduplication, or unmatched transaction users in this extract. It also found zero exact duplicate user rows.